# Intermediate Data Science

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209 -- [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/data201_intermediate.html)
- [Syllabus](https://joannabieri.com/data201/IntermediateDataScience.pdf)

## Today's Reading

*Python for Data Analysis*, Chapter 7 - Data Cleaning and Preparation.

## Career Reading (discuss Tuesday 9/15)

*Build a Career in Data Science*, 1.2 Different Types of Data Science Jobs. We will talk about it at the start of Tuesday's class, so come with a thought or two.

## Data Cleaning

Often in data science a huge portion of your time will be spent loading, cleaning, transforming, and rearranging data. Sometimes you will have data sets that contain many missing variables, others will have bad formatting such as numbers being read in as strings. Sometimes you will need to create dummy variables or introduce new variables into your data. Often you will need to take sub-samples of your data for training and testing a model or simply because the data is too big to read in all at one time. Fortunately, pandas and python have lots of tools to help you in this process. Here are the main topics we will cover:

1. Handling Missing Data
2. Transforming Data
3. String Manipulation
4. Categorical Data

In [ ]:
# Some basic package imports
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook_connected'

## Handling Missing Data

Missing data can happen for a wide variety of reasons:

- The data really does not exist for a certain observation: In population data babies would not have a date of marriage or a list of children.
- The data was improperly entered: Errors are easy to make.
- The data set was damaged: Reading or writing issues happen.
- The missing data (None, NaN, or NA) means something important: Maybe a student did not take a test and that is important in your analysis.

Pandas uses floating point NaN (Not a Number) to represent missing data.

In [ ]:
np.nan

In [ ]:
type(np.nan)

It will interpret NA (Not Available) as a Python None. This will be considered NaN when doing analysis.

In [ ]:
string_data = pd.Series(["aardvark", np.nan, None, "avocado"])
string_data

### Methods to handle missing data:

1. `isna()` / `isnull()`
- **Description:** Detect missing values.
  ```python
  df.isna()
  df['column'].isnull()
  ```

2. `notna()` / `notnull()`
- **Description:** Detect non-missing values.
  ```python
  df.notna()
  df['column'].notnull()
  ```

3. `dropna()`
- **Description:** Remove missing values.
  ```python
  df.dropna()  # Drop rows with any NA values
  df.dropna(axis=1)  # Drop columns with any NA values
  ```

4. `fillna()`
- **Description:** Fill NA values with a specified value or method.
  ```python
  df.fillna(0)  # Replace NA with 0
  ```

5. `replace()`
- **Description:** Replace specified values including NA.
  ```python
  df.replace(to_replace=np.nan, value=0)
  ```

In [ ]:
string_data.isna()

In [ ]:
string_data.notna()

In [ ]:
string_data.fillna(0)

In [ ]:
string_data.replace(np.nan,0)

You will notice that these methods do not write over the data in memory. In other words, if we print what is in string_data we will see all the nans are still there:

In [ ]:
string_data

If you want to make the changes in memory you need to add the flag: `inplace=True`, or reset the variable

```{python}
string_data.fillna(0,inplace=True)
string_data = string_data.fillna(0)
```

In [ ]:
string_data.fillna(0,inplace=True)

In [ ]:
string_data

### **BEWARE OF PYTHON mutable!**

When you are saving data from one list to the next you should be very careful about how you do that! Python lists are **mutable** this means that when you set one list equal to another it does not make a new copy in memory, instead it copies a reference. Here is an example:

In [ ]:
list1 = [5,4,3,2,1]
list2 = list1

print('Here is list2, it looks like a copy!')
print(list2)

print('Now we will change something in list2')
list2[0] = 10
print(list2)

print('Now look at list1')
print(list1)

print('BUT WE DIDN"T CHANGE LIST1 !!!!! WHY DID IT CHANGE????')

It changed because list1 and list2 point to the same object in memory. How do we stop this from happening? We need to use `.copy()` when making a copy of a list to get new memory allocated. Lets do the same computation but this time use .copy()

In [ ]:
list1 = [5,4,3,2,1]
list2 = list1.copy() ## THIS IS OUR ONLY CHANGE

print('Here is list2, it looks like a copy!')
print(list2)

print('Now we will change something in list2')
list2[0] = 10
print(list2)

print('Now look at list1')
print(list1)
print('List1 did not change')

### **Moral of the mutable story**

If you are creating a new variable by setting it equal to another list and you want to make changes to one without changing the other you should use .copy(). This is true of all mutable python types:

- list
- dict
- set
- pd.DataFrame
- pd.Series
- np.array

### Filtering out Missing Data

You want to be careful and intentional when filtering out missing data. We will explore the process with a DataFrame that contains lots of missing data.

In [ ]:
data = pd.DataFrame([[1., 6.5, 3.], [ np.nan, np.nan, 1.],
                     [np.nan, np.nan, np.nan], [np.nan, 6.5, 3.]])
data

In [ ]:
# We can drop all NaNs
data.dropna()

In [ ]:
# We can drop only rows that are completely NaNs
data.dropna(how='all')

In [ ]:
# We can drop only rows that contain more than two NaN's
data.dropna(thresh=2)

In [ ]:
# We can drop only columns that contain more than two NaN's
data.dropna(thresh=2,axis=1)

Notice that each of these decisions creates a very different result! You should also notice that some optional commands are pretty common:

- `axis=0` do the calculation to the rows - usually default
- `axis=1` do the calculation to the columns

In [ ]:
# data.dropna?

### Filling in Missing Data

Sometimes you want to replace missing data in a DataFrame in a managed way so that it has predictable effects on the rest of your analysis.
Most of the time you will use `.fillna()` but there are some nice optional arguments that let you customize the command. Remember, to make changes in memory you need to add `inplace=True`.

In [ ]:
data

In [ ]:
# Change NaN to zero
data.fillna(0)

In [ ]:
# Change each column differently - use a dictionary!
# Keys are column names
# Values are the fill 
data.fillna({0:np.nan,1:'Hello',2:0})

In [ ]:
# Fill with a calculation - here will will fill with the column means
data.fillna(data.mean())

## Data Transformation

Next we will talk a bit more about cleaning data:

1. Removing Duplicate Data
2. Replacing Data
3. Renaming
4. Discretizing and Binning
5. Outliers
6. Sampling
7. Dummy Variables

### Duplicate Data

Sometimes data sets will have duplicate variables, maybe they are not identical but they represent the same thing. Here column k1 has the words and k2 has the numerical values:

In [ ]:
data = pd.DataFrame({"k1": ["one", "two"] * 3 + ["two"],
                     "k2": [1, 1, 2, 3, 3, 4, 4]})
data

Notice that observations 5 and 6 are identical! We can check for this using the `.duplicated()` and `drop_duplicates()` command.

In [ ]:
data.duplicated()

In [ ]:
data.drop_duplicates()

You can filter duplicates on a subset of the data if you specify the rows that you want to filter. Lets add a column to our data and then drop duplicates.

In [ ]:
data["v1"] = range(7)
data['v2'] = [1 for i in range(7)]
data

In [ ]:
# We can drop any duplicates just in the k1 column - returns just the first two observations
# Notice we still get back all the other data in those two rows
data.drop_duplicates(subset=['k1'])

In [ ]:
# We can look for duplicates across two different columns
data.drop_duplicates(subset=['k2','v2'])

### Transforming Data and Mapping

We will imagine that you have some data that tells you about the amount of different types of fruit. Say you want to add a new column to this data that says what kind of fruit each one is: citrus, berry, tropical, or pome. You can use a dictionary and the `.map()` function to add this information.

In [ ]:
# Here is our made up data
data = pd.DataFrame({
    "food": ["orange", "blueberry", "orange",
             "banana", "strawberry", "orange",
             "banana", "apple", "blackberry"],
    "ounces": [4, -999, 12, 6, 7.5, 8, -999, 5, 6]
})

data

In [ ]:
# Here is a dictionary mapping fruits to categories
food_to_category = {
    "orange": "citrus",
    "blueberry": "berry",
    "strawberry": "berry",
    "blackberry": "berry",
    "banana": "tropical",
    "apple": "pome"
}

In [ ]:
# Now we will add the column
data['category'] = data['food'].map(food_to_category)
data

### Replacing Values

We have seen above how to replace NaN values, but what if there were other types of things you wanted to replace in the dataset? The `.replace()` function can replace any data you want with replacement data.  

In [ ]:
# Change the category given to banana
data.replace('tropical','berry',inplace=True)
data

In [ ]:
# Remove numbers that we know are wrong
# Sometimes NaN are coded with large negative or really unreasonable numbers
data.replace(-999,np.nan)

### Renaming

There are lots of ways to rename things on both the column labels and the index labels in a data frame. Here are a few examples of doing this.

In [ ]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=["Ohio", "Colorado", "New York"],
                    columns=["one", "two", "three", "four"])
data

In [ ]:
# We could change things directly
# Here we send in a dictionary that gives the label as a key and the new label as the value
# Columns
data.rename(columns = {'three':'THREE', 'four':'FOUR'}, inplace=True)
data

In [ ]:
# Rows
data.rename(index = {'Ohio':'California'}, inplace=True)
data

In [ ]:
# We could also define a function that will transform the data through a map
def new_names(x):
    return x[:4].upper()

data.index = data.index.map(new_names)
data

In [ ]:
# define a dictionary map
text_to_num = {'one':1,'two':2,'three':3,'four':4}
# define a function
def change_to_num(x):
    return text_to_num[x.lower()]

data.columns = data.columns.map(change_to_num)
data

### You Try

Run the cell below to get your data. Then update the column and index names, using the renaming methods above, so that they are consistent and easy to use. Your choice for how you want the final labels to be!

In [ ]:
data = pd.DataFrame(np.arange(12).reshape((3, 4)),
                    index=["red green", "Blue_green", "green  "],
                    columns=["ONE", "two", "3", "Four"])
data

In [ ]:
# Your code here

### Discretization and Binning

Sometimes you will want to take continuous data and represent it as bins. This is often done in a histogram, but in data science maybe you want to define categories based on a continuous numerical variable. For example, maybe you want high, medium, and low income classes. This is where binning will help.



In [ ]:
# Get the list of ages
ages = [20, 22, 25, 27, 21, 23, 37, 31, 61, 45, 41, 32]
# Choose the edges of  your bins
# Here we will do 18 and under, 19-24, 25-34, ....
bins = [18, 25, 35, 60, 100]
# Have pandas cut the data into bins
age_categories = pd.cut(ages, bins)
age_categories

In [ ]:
# Now put this data into a DataFrame
data = pd.DataFrame()
data['age'] = ages
data['range'] = age_categories
data

In [ ]:
# The object returned by .cut() has some other features
# You can look at the categories that were in the data set
age_categories.categories

In [ ]:
# You can also get category codes
age_categories.codes

In [ ]:
# Lets add the codes to the DataFrame
data['code'] = age_categories.codes
data

Notice that now we have both the range for the category and a code that puts them into discrete numerical groups.

In the range column the notation that you see is:

- `(` means that endpoint is NOT included
- `]` means that endpoint IS included

so you would read the range `(18, 25]` as ages over 18, up to and including 25.

In [ ]:
# You can also give pandas a number of bins to use and 
# it will compute equal length bins to put your data into
pd.cut(ages,4)

### You Try

Run the cell below to create a random list of numbers to represent ages in your population. Then make up your own age range categories (at least 5) and use `.cut()` to break the data into discrete categories. Create a data frame that contains the age, the age range, and the age category code.

In [ ]:
ages = [np.random.randint(15,100) for i in range(40)]

In [ ]:
# Your code here

### Detecting Outliers

Sometimes you want to be able to detect outliers in a dataset, however this process can take a variety of operations and is highly dependent on how you define outliers in your data. Here is an example data set that just randomly assigns values based on a normal distribution

In [ ]:
data = pd.DataFrame(np.random.standard_normal((1000, 1)))
data.describe()

In [ ]:
# You could write a function
# These can sometimes get complicated
def check_outlier(x,q1 = data.quantile(0.25)[0],q3 = data.quantile(0.75)[0]):
    '''
    This function calculates the quantiles and then applies the outlier 
    checker based on the interquartile range.

    It can only accept one column at a time.

    NOTE: When using a function as a map, you can only pass in one variable.
    '''
    IQR = q3 - q1
    # Determine bounds
    lower_bound = q1 - 1.5 * IQR
    upper_bound = q3 + 1.5 * IQR
    if x>upper_bound:
        return True
    elif x<lower_bound:
        return True
    else:
        return False

# And then apply a map
data['outlier']=data.map(check_outlier)
data

In [ ]:
# Check how many outliers we have
data['outlier'].value_counts()

In [ ]:
# Mask out the outliers
data = data[data['outlier'] != True]
data

In [ ]:
# you could also just choose an upper value
# then correct for anything outside that value
# here is our data
data = pd.DataFrame(np.random.standard_normal((1000, 4)))
display(data)

# here is our update
data[data.abs() > 1] = np.sign(data) *2
display(data)

### You Try

Explain in great detail what each of the lines in the above cell did to both create and then update the data frame.

In [ ]:
# Your WORDS here - change this cell to markdown

### Permutation and Random Sampling

Often when doing a data science project you will want to take random samples of your data. This might be to help you avoid bias in the ordering of your data. It might be to create model training and testing data sets. Or maybe your data set is too big and you want to start with a smaller subset of the data. There are lots of ways to do this:

- NUMPY - has `random.permutation()` which will give you a list of integers in a range that are permuted (rearranged) randomly.
- PANDAS - has a function `.sample()` that can take a sample from a DataFrame or series.
- Other Packages - later this semester we will see other packages like sklearn that can create test-train splits of your data.

In [ ]:
# Here is some fake data
df = pd.DataFrame(np.arange(5 * 7).reshape((5, 7)))
df

In [ ]:
# Notice how this data is really ordered - maybe this is not good for your analysis
# We will grab the rows in a different order randomly
num_rows = 5
sample = np.random.permutation(num_rows)
print(sample)

# Now get the rows in that order
df.take(sample)

In [ ]:
# We can also graph the columns in a different order
num_cols = df.shape[1] 
sample = np.random.permutation(num_cols)
print(sample)

# Now get the columns in that order
df.take(sample, axis=1)

In [ ]:
# From within pandas we can get a sub-sample of our data
df.sample(n=3)

In [ ]:
# Sometimes you don't mind choosing the same row twice
# replace=True lets you sample the same row more than once
df.sample(n=30, replace=True)

### Dummy Variables

Dummy variables are variables that take the place of something in your data set. They are especially useful for classifying categorical data. In these cases you replace a column with categories with several columns of 0 or 1 to represent whether or not (True/False) the observation belongs to the category. Here is an example:

Lets say we have some categorical data that we want to interact with numerically. Below you will see the key column that contains a,b,c. Lets generate dummy variables for this data.



In [ ]:
df = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"],
                   "data1": range(6)})
df

In [ ]:
dummies = pd.get_dummies(df["key"])
dummies

In [ ]:
dummies = pd.get_dummies(df["key"],dtype=float)
dummies

In [ ]:
dummies = pd.get_dummies(df["key"],dtype=int)
dummies

In [ ]:
# Now add this to the original data frame
df_new = df.join(dummies)
df_new

Now we have three new columns each of which represents a true/false for whether or not the data had that key.

**Here is a slightly more complicated example.**

*These files contain 1,000,209 anonymous ratings of approximately 3,900 movies 
made by 6,040 MovieLens users who joined MovieLens in 2000. Thanks to Shyong Lam and Jon Herlocker for cleaning up and generating the data set. See README for more information*

In [ ]:
mnames = ["movie_id", "title", "genres"]
movies = pd.read_table("data/movielens/movies.dat", sep="::",
                       header=None, names=mnames, engine="python")
movies[:10]

Notice how we have a bunch of genres. Maybe we want to break this up into categories that can be used mathematically - meanings we need numbers instead of words. The pandas `.get_dummies()` command can help us here!

First lets get the counts for the different category combinations:

In [ ]:
movies['genres'].value_counts()

Notice how there are 301 different categories. Why so many? Well if we look at the names, some movies are part of multiple categories and those categories are split up by the | character. How could we find out the number of Genres?

In [ ]:
# I will do this by writing a quick for loop
genre_set = set()
for g in movies['genres']:
    for n in g.split('|'):
        genre_set.add(n)

print(len(genre_set))
genre_set

Okay so really there are 18 total genres. Let's use pandas to break this into dummy variables.

In [ ]:
dummies = movies['genres'].str.get_dummies("|")
dummies

Notice that this looks at the 'genres' element, breaks up the string by the | character, and then assigns it a 1 in any category that it belongs to. If we look at the first row, which represents Toy Story, we can see this movie belongs to the categories: Animation, Children's, Comedy.

Let's add these dummy variables to our original DataFrame

In [ ]:
# The .add_prefix() function adds a name to the beginning of the column
# This helps specify which variable the dummy is representing
movies_new = movies.join(dummies.add_prefix('genre_'))
movies_new

## String Manipulation

One of the most common things you will have to do is interact with data that is formatted as strings (words). This can happen because of the way you saved your data, maybe everything got turned into strings, or because of the way the data was originally entered.

1. Simple strings to numbers
2. Object methods

Anything that is entered into python with quotes or read in as a string will be assumed to be a string. Even though the number below is clearly, to our human minds, a number, Python sees it as a word.

In [ ]:
number = '3'
number

In [ ]:
# We can change it into an integer
int(number)

In [ ]:
# We can change it into a float
float(number)

In [ ]:
# We can change a number back into a string
str(3.0)

If we start with a more complicated string there are lots of things we can do to alter it. The `.split()` function can split up a string how ever you want! It turns the string into a list of substrings.

In [ ]:
val = "a,b,  guido"
val.split(",")

In [ ]:
val.split('  ')

In [ ]:
# Notice how the extra whitespace is annoying here!
string_list = val.split(',')
print(string_list)

# The .strip() will get rid of any white space
# Here is an example of a for loop inside a list
# This is called list comprehension
new_string_list = [x.strip() for x in string_list]
print(new_string_list)

In [ ]:
# Maybe we then want to put these pieces together
# the .join() command will cycle through the list and join up the strings
# with the given string
'.'.join(new_string_list)

In [ ]:
# You can check for membership
print('guido' in new_string_list)
print('g' in 'guido')

In [ ]:
# You can find things inside a string
my_string = 'Hello World'
print(my_string.find('W'))
my_string[6]

## You Try

Break the following string up into a list of strings using string manipulation functions. See if you can create a list like this:

    ['Joanna','Bieri','Redlands','Keep up the good work!']

try to get all the capitals and spacing correct!

NOTE - lots of different processes will result in this final list, there is not one right way to do this!

In [ ]:
a_string = 'joanna_bieri@redlands.edu says:   Keep up the good work!'

In [ ]:
# Your code here

There are so many different string methods! Here are some of my favorites:

- `.replace(old text, new text)` replace text
- `.rstrip()` strip from the right end of the string
- `.lstrip()` strip from the left end of the string
- `.lower()` make the string all lower case
- `.upper()` make the string all upper case
- `.title()` capitalize each first letter

For more advanced string manipulation you can use regex.

    import re

look in the book or online for more information.

Many of the string methods are also implemented directly in pandas and can be applied directly to Series data.

## Categorical Data

Pandas has a built-in data type called Categorical. This helps encode certain columns or parts of your data as being categorical, rather than just an assortment of strings. Categorical has the advantage of being better for memory and for sorting and organizing data. Here is an example:

In [ ]:
# Simulate a column with repeated strings
# Do not turn the data into categorical - keep it as strings
n = 1_000_000
df = pd.DataFrame({
    'city': np.random.choice(['New York', 'Los Angeles', 'Chicago'], size=n)
})

df

In [ ]:
# Now create a column with the same data, but tell pandas it is categorical
df['city_cat'] = df['city'].astype('category')
df

In [ ]:
# Look at the memory usage of each column
print(df['city'].memory_usage(deep=True))
print(df['city_cat'].memory_usage(deep=True))

In [ ]:
# Lets get the list of cities and order them reverse alphabetically
levels = list(df['city'].value_counts().keys().sort_values(ascending=False))
levels

In [ ]:
# Now create a categorical data column that includes the levels as categories
# and tells pandas that the categories are ordered.
df['city_cat_levels'] = pd.Categorical(df['city'], categories=levels, ordered=True)
df

In [ ]:
# Now because we called our city_cat_levels column categorical and assigned levels
# Now we can compare using < and >
# And assign an ordering to our data
print(df['city_cat_levels'] > 'Los Angeles')  # True for 'Chicago'

In [ ]:
# The standard comparison only gives you the option to check alphabetically
print(df['city'] > 'Los Angeles')

## Homework 4

The full assignment is in `HW_day4.ipynb` in your team repo. You will download a
deliberately messy Kaggle dataset and put today's tools to work on it: missing values,
duplicates, mapping a category column, binning, dummy variables, stripping junk
characters out of the money columns, and splitting one column into two.

Work in a branch, commit as you go, and open a Pull Request into `main` so a teammate
can review it before it gets merged. HW 3 and HW 4 are both due Sunday 9/13 by 11:59pm.
Nothing gets uploaded to Canvas, but do not forget your individual weekly reflection
over there.